# Fig 5 power vs FDR: SuSiE-RSS vs SuSiE-rRSS against in-sample PIPs (data processing)

From `35.Fig5_PR_ROC_Extract` (`2026_output/Fig5_PR_ROC/result/`): per dataset (GTEx eQTL, Fig 5A; UKB GWAS, Fig 5C) with the ADSP sketch reference (`ld_ref = "ADSP"`), or the 1000 Genomes reference (`ld_ref = "1000G"`, Fig S11A / S11B), the first `n_units` = 2000 eligible units in the random order of `[pr_units]`. A unit (gene × tissue, or region × trait) is eligible when the in-sample fit, SuSiE-RSS or SuSiE-rRSS (after SER fallback) has a credible set. Truth fit: GTEx individual-level SuSiE, UKB SuSiE-RSS with in-sample LD (the in-sample fits of Fig 5). SuSiE-rRSS = full process (SER model when flagged).

Variant level, all units pooled:
- **truth:** a variant is a true signal when its in-sample PIP ≥ `t`, for `t` = 0.5, 0.7, 0.8 (three sets of curves).
- **curve:** variants called at a PIP threshold = method PIP ≥ threshold; power = true signals called / all true signals, FDR = 1 − true signals called / variants called. The curve traces all thresholds down to `pip_floor` = 0.01 (below it the FDR is above 0.98); it is not monotone: when a method's most confident calls are mostly false, the FDR falls as the threshold is lowered.
- **points:** PIP thresholds 0.05, 0.10, …, 0.95, each with bootstrap 95% CIs of its FDR and power (1000 resamples of genes (GTEx) or regions (UKB), all units of a drawn gene / region together). They lie on the curve.
- **power at FDR ≤ f** (`f` = 0.05, 0.10, 0.20; text only): the largest power among thresholds with FDR ≤ f (0 if none), point estimate and threshold. No CI: it jumps where the curve has a cliff, so its bootstrap distribution is bimodal.

**Outputs** (1000 Genomes: `Fig5_Power_FDR_1000G_*`):
- `figure_data/Fig5_Power_FDR_Curves.rds` (+ `.csv`): dataset × truth × method × threshold: FDR, power (thinned for plotting).
- `figure_data/Fig5_Power_FDR_Points.rds` (+ `.csv`): dataset × truth × method × PIP threshold: FDR and power with bootstrap 95% CIs, variants called, true signals.
- `figure_data/Fig5_Power_FDR_AtFDR.rds` (+ `.csv`): dataset × truth × method × FDR level: power and the PIP threshold that reaches it.
- `figure_data/Fig5_Power_FDR_Units.rds` (+ `.csv`): the units used.

Fast (minutes); run interactively, or as a job with `35` section 4.

In [12]:
source("../../R/utils.R")

ld_ref     <- Sys.getenv("LD_REF", "ADSP")   # "ADSP" (Fig 5A / 5B) or "1000G" (Fig S11A / S11B); the job in 35 section 4 sets LD_REF
sfx        <- if (ld_ref == "1000G") "_1000G" else ""
data_name  <- paste0("Fig5_Power_FDR", sfx)
root       <- Sys.getenv("PR_ROOT", "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fig5_PR_ROC/result")   # 35 [pr_extract]
n_units    <- 2000
B          <- 1000
n_cores    <- as.integer(Sys.getenv("LSB_DJOB_NUMPROC", "4"))    # bootstrap in parallel (forked)
seed       <- 2026
truth_thr  <- c(0.5, 0.7, 0.8)               # true signal: in-sample PIP >= truth_thr
pip_grid   <- seq(0.05, 0.95, by = 0.05)     # points with bootstrap CIs (FDR and power)
fdr_levels <- c(0.05, 0.1, 0.2)              # power at FDR <= f (text)
pip_floor  <- 0.01                           # the curve: thresholds down to pip_floor
datasets   <- c(GTEx = "GTEx eQTL", UKB = "UKB GWAS")
methods    <- c(pip_rss = "SuSiE-RSS", pip_rrss = "SuSiE-rRSS")
truth_lab  <- function(t) sprintf("In-sample PIP ≥ %.1f", t)

read_dataset <- function(ds) {
  ch <- list.files(root, pattern = paste0("^", ds, sfx, "_pr_chunk_[0-9]+\\.rds$"), full.names = TRUE)
  stopifnot(length(ch) > 0)
  x  <- map(ch, readRDS)
  min_pip <- unique(map_dbl(x, "min_pip"))
  stopifnot("35 [pr_extract] must keep every variant (min_pip = 0)" = identical(min_pip, 0))
  unit <- map_dfr(x, "unit"); var <- bind_rows(map(x, "var")); rm(x)
  cat(sprintf("\n== %s%s: %d chunks, %d units read\n", ds, sfx, length(ch), nrow(unit)))
  print(count(unit, status, eligible))
  keep <- unit %>% filter(status == "ok", eligible) %>% arrange(rank) %>% slice_head(n = n_units)
  if (nrow(keep) < n_units) warning(ds, ": only ", nrow(keep), " eligible units (raise oversample in 35 [pr_units])")
  keep <- keep %>% mutate(dataset = datasets[[ds]], u = row_number())
  var  <- var %>% inner_join(select(keep, rank, u), by = "rank") %>% select(u, pip_truth, pip_rss, pip_rrss)
  list(unit = keep, var = var)
}
dat <- map(names(datasets), read_dataset) %>% set_names(names(datasets))
map_dfr(dat, ~ .x$unit %>% summarise(dataset = first(dataset), units = n(), genes_or_regions = n_distinct(gene),
                                     pct_ser_fallback = 100 * mean(ser_fallback), with_insample_cs = sum(n_cs_truth > 0),
                                     variants = nrow(.x$var),
                                     signals_05 = sum(.x$var$pip_truth >= 0.5), signals_07 = sum(.x$var$pip_truth >= 0.7),
                                     signals_08 = sum(.x$var$pip_truth >= 0.8)))


== GTEx_1000G: 40 chunks, 2600 units read
# A tibble: 2 × 3
  status eligible     n
  <chr>  <lgl>    <int>
1 ok     FALSE        1
2 ok     TRUE      2599

== UKB_1000G: 40 chunks, 2600 units read
# A tibble: 2 × 3
  status eligible     n
  <chr>  <lgl>    <int>
1 ok     FALSE        4
2 ok     TRUE      2596


dataset,units,genes_or_regions,pct_ser_fallback,with_insample_cs,variants,signals_05,signals_07,signals_08
<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>
GTEx eQTL,2000,1347,22.00,1933,7593852,911,644,535
UKB GWAS,2000,775,4.25,1910,4393527,955,630,521


In [13]:
## ---- power / FDR curves -----------------------------------------------------------------------
## one method's variants with PIP >= floor, ranked by PIP (decreasing); y = true signal (0 / 1)
## at = number of ranked variants with PIP >= each grid threshold
rank_prep <- function(s, y, u, floor, grid) {
  k <- which(s >= floor); o <- k[order(-s[k])]; s <- s[o]
  list(s = s, y = y[o], u = u[o], last = which(c(s[-1] != s[-length(s)], TRUE)),   # last variant of each tie block
       at = vapply(grid, function(g) sum(s >= g), integer(1)))
}
## FDR and power at every distinct threshold (cnt = bootstrap multiplicity of each unit, pos_u = true signals per unit)
pf_curve <- function(pr, pos_u, cnt) {
  cw <- cnt[pr$u]
  tp <- cumsum(cw * pr$y)[pr$last]; n <- cumsum(cw)[pr$last]; k <- n > 0
  tibble(threshold = pr$s[pr$last][k], fdr = 1 - tp[k] / n[k], power = tp[k] / sum(cnt * pos_u))
}
## FDR, power and variants called at the grid thresholds (NA FDR when nothing is called)
pf_grid <- function(pr, pos_u, cnt) {
  cw <- cnt[pr$u]
  tp <- c(0, cumsum(cw * pr$y))[pr$at + 1]; n <- c(0, cumsum(cw))[pr$at + 1]
  list(fdr = ifelse(n > 0, 1 - tp / n, NA_real_), power = tp / sum(cnt * pos_u), called = n)
}
## power at FDR <= f: the largest power among thresholds with FDR <= f (0 if none), and that threshold
power_at <- function(cv, f) vapply(f, function(x) max(c(0, cv$power[cv$fdr <= x + 1e-12])), numeric(1))
thr_at   <- function(cv, f) vapply(f, function(x) {
  ok <- which(cv$fdr <= x + 1e-12); if (length(ok)) cv$threshold[ok[which.max(cv$power[ok])]] else NA_real_ }, numeric(1))
## bootstrap multiplicities: genes / regions drawn with replacement, all their units together (units x B)
boot_counts <- function(cluster, B, seed) {
  cl <- as.integer(factor(cluster)); K <- max(cl)
  set.seed(seed)
  vapply(seq_len(B), function(b) tabulate(sample.int(K, K, replace = TRUE), K)[cl], numeric(length(cl)))
}
## thin a curve for plotting: one point per rounded (FDR, power)
thin_curve <- function(cv) cv %>% distinct(f = round(fdr, 3), p = round(power, 3), .keep_all = TRUE) %>% select(threshold, fdr, power)
ci <- function(m, q) apply(m, 2, quantile, q, names = FALSE, na.rm = TRUE)

In [14]:
curves <- list(); pts <- list(); atf <- list()
for (ds in names(dat)) {
  d <- dat[[ds]]; un <- d$unit; v <- d$var; nu <- nrow(un)
  cnt_B <- boot_counts(un$gene, B, seed)
  for (t in truth_thr) {
    y     <- as.numeric(v$pip_truth >= t)
    pos_u <- tabulate(v$u[y == 1], nu)
    for (m in names(methods)) {
      pr  <- rank_prep(v[[m]], y, v$u, pip_floor, pip_grid)
      cv  <- pf_curve(pr, pos_u, rep(1, nu))
      est <- pf_grid(pr, pos_u, rep(1, nu))
      bt  <- parallel::mclapply(seq_len(B), function(b) pf_grid(pr, pos_u, cnt_B[, b])[c("fdr", "power")], mc.cores = n_cores)
      bf  <- do.call(rbind, map(bt, "fdr")); bp <- do.call(rbind, map(bt, "power"))      # B x grid
      lab <- tibble(dataset = datasets[[ds]], ld_ref = ld_ref, truth = truth_lab(t), method = methods[[m]])
      curves[[length(curves) + 1]] <- bind_cols(lab, thin_curve(cv))
      pts[[length(pts) + 1]] <- bind_cols(lab, tibble(
        pip_threshold = pip_grid, fdr = est$fdr, fdr_lo = ci(bf, 0.025), fdr_hi = ci(bf, 0.975),
        power = est$power, power_lo = ci(bp, 0.025), power_hi = ci(bp, 0.975),
        called = est$called, n_signals = sum(pos_u), units = nu, genes_or_regions = n_distinct(un$gene),
        pct_ser_fallback = 100 * mean(un$ser_fallback)))
      atf[[length(atf) + 1]] <- bind_cols(lab, tibble(fdr_level = fdr_levels, power = power_at(cv, fdr_levels),
                                                      pip_threshold = thr_at(cv, fdr_levels)))
      i5 <- which.min(abs(pip_grid - 0.5)); i95 <- which.min(abs(pip_grid - 0.95))
      cat(sprintf("%s, truth PIP >= %.1f, %s: PIP >= 0.5: FDR %.3f, power %.3f | PIP >= 0.95: FDR %.3f, power %.3f | power at FDR 0.05: %.3f\n",
                  ds, t, m, est$fdr[i5], est$power[i5], est$fdr[i95], est$power[i95], power_at(cv, 0.05)))
    }
  }
}
lv <- function(x) x %>% mutate(dataset = factor(dataset, levels = datasets), truth = factor(truth, levels = truth_lab(truth_thr)),
                               method = factor(method, levels = methods))
curves <- lv(bind_rows(curves)); pts <- lv(bind_rows(pts)); atf <- lv(bind_rows(atf))

pts %>% filter(abs(pip_threshold - 0.5) < 1e-9 | abs(pip_threshold - 0.95) < 1e-9) %>%
  select(dataset, truth, method, pip_threshold, fdr, fdr_lo, fdr_hi, power, power_lo, power_hi, called, n_signals) %>%
  mutate(across(where(is.double), ~ signif(.x, 3))) %>% as.data.frame()
atf %>% mutate(across(where(is.double), ~ signif(.x, 3))) %>% as.data.frame()

units_used <- map_dfr(dat, "unit") %>% select(-u)
save_data(curves,     paste0(data_name, "_Curves"))
save_data(pts,        paste0(data_name, "_Points"))
save_data(atf,        paste0(data_name, "_AtFDR"))
save_data(units_used, paste0(data_name, "_Units"))

GTEx, truth PIP >= 0.5, pip_rss: PIP >= 0.5: FDR 0.378, power 0.693 | PIP >= 0.95: FDR 0.371, power 0.318 | power at FDR 0.05: 0.000
GTEx, truth PIP >= 0.5, pip_rrss: PIP >= 0.5: FDR 0.100, power 0.605 | PIP >= 0.95: FDR 0.062, power 0.234 | power at FDR 0.05: 0.059
GTEx, truth PIP >= 0.7, pip_rss: PIP >= 0.5: FDR 0.493, power 0.800 | PIP >= 0.95: FDR 0.386, power 0.439 | power at FDR 0.05: 0.000
GTEx, truth PIP >= 0.7, pip_rrss: PIP >= 0.5: FDR 0.242, power 0.720 | PIP >= 0.95: FDR 0.062, power 0.331 | power at FDR 0.05: 0.084
GTEx, truth PIP >= 0.8, pip_rss: PIP >= 0.5: FDR 0.570, power 0.815 | PIP >= 0.95: FDR 0.395, power 0.521 | power at FDR 0.05: 0.000
GTEx, truth PIP >= 0.8, pip_rrss: PIP >= 0.5: FDR 0.348, power 0.746 | PIP >= 0.95: FDR 0.062, power 0.398 | power at FDR 0.05: 0.101
UKB, truth PIP >= 0.5, pip_rss: PIP >= 0.5: FDR 0.540, power 0.747 | PIP >= 0.95: FDR 0.501, power 0.372 | power at FDR 0.05: 0.000
UKB, truth PIP >= 0.5, pip_rrss: PIP >= 0.5: FDR 0.163, power 0.593

dataset,truth,method,pip_threshold,fdr,fdr_lo,fdr_hi,power,power_lo,power_hi,called,n_signals
<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
GTEx eQTL,In-sample PIP ≥ 0.5,SuSiE-RSS,0.50,0.3780,0.3200,0.4370,0.693,0.657,0.725,1020,911
GTEx eQTL,In-sample PIP ≥ 0.5,SuSiE-RSS,0.95,0.3710,0.2790,0.4610,0.318,0.284,0.354,461,911
GTEx eQTL,In-sample PIP ≥ 0.5,SuSiE-rRSS,0.50,0.0997,0.0746,0.1250,0.605,0.571,0.641,612,911
GTEx eQTL,In-sample PIP ≥ 0.5,SuSiE-rRSS,0.95,0.0617,0.0286,0.1000,0.234,0.203,0.267,227,911
GTEx eQTL,In-sample PIP ≥ 0.7,SuSiE-RSS,0.50,0.4930,0.4380,0.5430,0.800,0.763,0.832,1020,644
GTEx eQTL,In-sample PIP ≥ 0.7,SuSiE-RSS,0.95,0.3860,0.2990,0.4770,0.439,0.398,0.480,461,644
GTEx eQTL,In-sample PIP ≥ 0.7,SuSiE-rRSS,0.50,0.2420,0.2070,0.2760,0.720,0.683,0.760,612,644
GTEx eQTL,In-sample PIP ≥ 0.7,SuSiE-rRSS,0.95,0.0617,0.0286,0.1000,0.331,0.290,0.373,227,644
GTEx eQTL,In-sample PIP ≥ 0.8,SuSiE-RSS,0.50,0.5700,0.5250,0.6170,0.815,0.776,0.848,1020,535


dataset,ld_ref,truth,method,fdr_level,power,pip_threshold
<fct>,<chr>,<fct>,<fct>,<dbl>,<dbl>,<dbl>
GTEx eQTL,1000G,In-sample PIP ≥ 0.5,SuSiE-RSS,0.05,0.0000,NA
GTEx eQTL,1000G,In-sample PIP ≥ 0.5,SuSiE-RSS,0.10,0.0000,NA
GTEx eQTL,1000G,In-sample PIP ≥ 0.5,SuSiE-RSS,0.20,0.0000,NA
GTEx eQTL,1000G,In-sample PIP ≥ 0.5,SuSiE-rRSS,0.05,0.0593,1.000
GTEx eQTL,1000G,In-sample PIP ≥ 0.5,SuSiE-rRSS,0.10,0.6080,0.500
GTEx eQTL,1000G,In-sample PIP ≥ 0.5,SuSiE-rRSS,0.20,0.6760,0.432
GTEx eQTL,1000G,In-sample PIP ≥ 0.7,SuSiE-RSS,0.05,0.0000,NA
GTEx eQTL,1000G,In-sample PIP ≥ 0.7,SuSiE-RSS,0.10,0.0000,NA
GTEx eQTL,1000G,In-sample PIP ≥ 0.7,SuSiE-RSS,0.20,0.0000,NA
